# Fine-tune Sarvam-1 on Colab

Sarvam-1 is a 2B completion model. This notebook LoRA-tunes `sarvamai/sarvam-1` on the poem JSONL. Ollama cannot do this step, and the local machine has no NVIDIA GPU.

Before running:

1. Upload this notebook to [Google Colab](https://colab.research.google.com/).
2. **Runtime → Change runtime type → T4 GPU**.
3. Run the cells from the top. The second cell mounts Google Drive and, the first time only, asks you to upload `data/sft/train.jsonl` (5.7 MB) and `data/sft/valid.jsonl` (0.6 MB). Those files stay in `My Drive/tagore-sft/`.
4. Training writes a checkpoint every 50 steps to that same Drive folder. If the session dies, rerun the notebook from the top with the same `LIMIT`. It continues from the latest checkpoint.
5. When training finishes, the last cell downloads `sarvam1-tagore-lora.zip`.

Set `LIMIT = 32` for a short smoke test. `LIMIT = None` uses the full training file. Do not change `LIMIT`, `EPOCHS`, or `MAX_LENGTH` while a checkpoint exists. One epoch is the first real run. A T4 takes a few hours on the full file.


In [ ]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "peft", "accelerate"])

import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if not torch.cuda.is_available():
    raise RuntimeError("Turn on a GPU runtime: Runtime → Change runtime type → T4 GPU, then rerun this cell.")


In [ ]:
import json
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DATA = Path("/content/drive/MyDrive/tagore-sft")
DRIVE_CKPT = DATA / "checkpoints"
ADAPTER_DIR = Path("/content/sarvam1-tagore-lora")
HF_MODEL = "sarvamai/sarvam-1"
MAX_LENGTH = 1536
EPOCHS = 1
LIMIT = None  # set to 32 for a smoke test
SAVE_STEPS = 50

DATA.mkdir(parents=True, exist_ok=True)
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

needed = ["train.jsonl", "valid.jsonl"]
if not all((DATA / name).exists() for name in needed):
    from google.colab import files
    print("Upload train.jsonl and valid.jsonl from data/sft/. They are copied to Google Drive.")
    uploaded = files.upload()
    for name, blob in uploaded.items():
        (DATA / Path(name).name).write_bytes(blob)

missing = [name for name in needed if not (DATA / name).exists()]
if missing:
    raise RuntimeError(f"Missing {missing}. Upload both jsonl files.")


def latest_checkpoint(root):
    found = []
    for path in root.glob("checkpoint-*"):
        if not (path / "trainer_state.json").exists():
            continue
        step = path.name.split("-")[-1]
        if step.isdigit():
            found.append((int(step), path))
    return max(found)[1] if found else None


run_config = {"limit": LIMIT, "epochs": EPOCHS, "max_length": MAX_LENGTH, "model": HF_MODEL}
config_path = DRIVE_CKPT / "run_config.json"
saved = latest_checkpoint(DRIVE_CKPT)
if saved and config_path.exists():
    previous = json.loads(config_path.read_text(encoding="utf-8"))
    if previous != run_config:
        raise RuntimeError(
            f"Drive checkpoint used {previous}, this run uses {run_config}. "
            "Keep LIMIT, EPOCHS, and MAX_LENGTH the same, or delete "
            "My Drive/tagore-sft/checkpoints to start over."
        )
else:
    config_path.write_text(json.dumps(run_config), encoding="utf-8")

RESUME = None
if saved:
    local = ADAPTER_DIR / saved.name
    if local.exists():
        shutil.rmtree(local)
    shutil.copytree(saved, local)
    RESUME = str(local)
    print("resuming from", saved)
else:
    print("no Drive checkpoint; training starts at step 0")
print("data", DATA)


In [ ]:
def load_jsonl(path):
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def prompt_and_poem(row):
    messages = row["messages"]
    user = next(item["content"] for item in messages if item["role"] == "user")
    poem = next(item["content"] for item in messages if item["role"] == "assistant")
    return user.strip() + "\n\n", poem


train_rows = load_jsonl(DATA / "train.jsonl")
valid_rows = load_jsonl(DATA / "valid.jsonl")
if LIMIT:
    train_rows = train_rows[:LIMIT]
    valid_rows = valid_rows[: max(8, LIMIT // 5)]
print(f"train {len(train_rows)}  valid {len(valid_rows)}")
sample_prompt, sample_poem = prompt_and_poem(train_rows[0])
print(sample_prompt)
print("\n".join(sample_poem.splitlines()[:4]))


## Train

Loss is on the poem tokens only. Poems that do not fit in 1536 tokens are skipped, so the model is not taught to stop mid-poem. A T4 does not support bf16, so training uses fp16.

Every 50 steps the finished checkpoint is copied to `My Drive/tagore-sft/checkpoints`. A later run of this notebook loads that copy and continues.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainerCallback, TrainingArguments
from peft import LoraConfig, get_peft_model


class SyncCheckpoint(TrainerCallback):
    """Copy a finished checkpoint to Drive. A crash mid-copy leaves the previous one intact."""

    def on_save(self, args, state, control, **kwargs):
        src = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if not (src / "trainer_state.json").exists():
            return
        dest = DRIVE_CKPT / src.name
        tmp = DRIVE_CKPT / f".{src.name}.partial"
        if tmp.exists():
            shutil.rmtree(tmp)
        shutil.copytree(src, tmp)
        if dest.exists():
            shutil.rmtree(dest)
        tmp.rename(dest)
        kept = sorted(
            (int(path.name.split("-")[-1]), path)
            for path in DRIVE_CKPT.glob("checkpoint-*")
            if path.name.split("-")[-1].isdigit()
        )
        for _, path in kept[:-2]:
            shutil.rmtree(path)
        print("checkpoint on Drive", dest)

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(HF_MODEL, torch_dtype=dtype)
model.config.use_cache = False

linear_names = sorted({
    name.split(".")[-1]
    for name, module in model.named_modules()
    if isinstance(module, torch.nn.Linear) and name.split(".")[-1] != "lm_head"
})
print("LoRA targets", linear_names)
model = get_peft_model(model, LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=linear_names,
))
model.print_trainable_parameters()
model.gradient_checkpointing_enable()
model.enable_input_require_grads()


In [ ]:
eos = tokenizer.eos_token or ""

def encode_example(row):
    prompt, poem = prompt_and_poem(row)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    poem_ids = tokenizer(poem + eos, add_special_tokens=False)["input_ids"]
    bos = [tokenizer.bos_token_id] if tokenizer.bos_token_id is not None else []
    input_ids = bos + prompt_ids + poem_ids
    if len(input_ids) > MAX_LENGTH:
        return None
    labels = [-100] * (len(bos) + len(prompt_ids)) + poem_ids
    return {"input_ids": input_ids, "labels": labels, "attention_mask": [1] * len(input_ids)}


def encode_split(rows):
    encoded, skipped = [], 0
    for row in rows:
        item = encode_example(row)
        if item is None:
            skipped += 1
        else:
            encoded.append(item)
    return encoded, skipped


class EncodedPoems(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


class PoemCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        width = max(len(feature["input_ids"]) for feature in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for feature in features:
            pad = width - len(feature["input_ids"])
            batch["input_ids"].append(feature["input_ids"] + [self.pad_token_id] * pad)
            batch["attention_mask"].append(feature["attention_mask"] + [0] * pad)
            batch["labels"].append(feature["labels"] + [-100] * pad)
        return {key: torch.tensor(value) for key, value in batch.items()}


train_encoded, train_skipped = encode_split(train_rows)
valid_encoded, valid_skipped = encode_split(valid_rows[:64])
print(f"train kept {len(train_encoded)} skipped {train_skipped}")
print(f"valid kept {len(valid_encoded)} skipped {valid_skipped}")

use_bf16 = torch.cuda.is_bf16_supported()
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=str(ADAPTER_DIR),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        bf16=use_bf16,
        fp16=not use_bf16,
        gradient_checkpointing=True,
        report_to="none",
        remove_unused_columns=False,
    ),
    train_dataset=EncodedPoems(train_encoded),
    eval_dataset=EncodedPoems(valid_encoded),
    data_collator=PoemCollator(tokenizer.pad_token_id),
    callbacks=[SyncCheckpoint()],
)
trainer.train(resume_from_checkpoint=RESUME)
final = DATA / "adapter"
if final.exists():
    shutil.rmtree(final)
trainer.save_model(str(final))
tokenizer.save_pretrained(str(final))
print("adapter on Drive", final)


In [ ]:
THEMES = [
    "বিষয়: বৃষ্টি\nএই বিষয়ে একটি কবিতা লেখো।",
    "নদী নিয়ে একটি কবিতা লেখো।",
    "প্রসঙ্গ: সোনার তরী, 1894। ১৮৯৪। পদ্মার বোটে জমিদারি জীবন। বিষয়: সন্ধ্যা। একটি কবিতা লেখো।",
]

model.eval()

def adapter_generate(theme, max_new_tokens=400):
    prompt = theme.strip() + "\n\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


samples = []
for theme in THEMES:
    text = adapter_generate(theme)
    samples.append({"prompt": theme, "completion": text})
    print("=" * 40)
    print(theme)
    print("-" * 40)
    print(text)
    print()

(DATA / "colab_samples.jsonl").write_text(
    "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in samples),
    encoding="utf-8",
)


In [ ]:
import shutil
from google.colab import files

zip_source = DATA / "adapter" if (DATA / "adapter").exists() else ADAPTER_DIR
zip_path = shutil.make_archive("/content/sarvam1-tagore-lora", "zip", zip_source)
print("download", zip_path)
files.download(zip_path)
